<a href="https://colab.research.google.com/github/metalBigLesson/PythonChess/blob/main/Chess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
from enum import Enum

def rank(rank, color_sign):
    if color_sign == 1:
        return rank - 1
    else:
        return 8 - rank

class Game:
    def __init__(self):
        self.turn_color_sign = -1
        self.display = [['♖','♘','♗','♕','♔','♗','♘','♖'],
                        ['♙','♙','♙','♙','♙','♙','♙','♙'],
                        ['□','■','□','■','□','■','□','■'],
                        ['■','□','■','□','■','□','■','□'],
                        ['□','■','□','■','□','■','□','■'],
                        ['■','□','■','□','■','□','■','□'],
                        ['♟','♟','♟','♟','♟','♟','♟','♟'],
                        ['♜','♞','♝','♛','♚','♝','♞','♜']]
        self.board = []
        for y in range(8):
            self.board.append([])
            for x in range(8):
                self.board[y].append(Empty(x, y))
        self.undo_move()
        self.end_turn()

    def get_piece(self, x, y):
        return self.board[y][x]

    def input_get(self):
        input_string = input("백 기물 선택: " if self.turn_color_sign == 1 else "흑 기물 선택: ")
        if len(input_string) != 2: raise Exception('잘못된 입력.')
        x = ord(input_string[0]) - ord('a')
        y = int(input_string[1]) - 1
        piece = self.get_piece(x, y)
        return piece

    def input_move(self):
        input_string = input("이동할 칸 선택: ")
        if len(input_string) != 2: raise Exception('잘못된 입력.')
        x = ord(input_string[0]) - ord('a')
        y = int(input_string[1]) - 1
        return x, y

    def is_self_check(self): return False#자살수 감지

    def undo_move(self):
        for i in range(8):
            for j in range(8):
                char = self.display[j][i]
                piece_char = str(self.get_piece(i, j))
                if char == piece_char: continue
                if (char == '□' or char == '■') and piece_char == ' ': continue
                match char:
                    case '♙': self.board[j][i] = Pawn(i, j, 1)
                    case '♟': self.board[j][i] = Pawn(i, j, -1)
                    case '♖': self.board[j][i] = Rook(i, j, 1)
                    case '♜': self.board[j][i] = Rook(i, j, -1)
                    case '♘': self.board[j][i] = Knight(i, j, 1)
                    case '♞': self.board[j][i] = Knight(i, j, -1)
                    case '♗': self.board[j][i] = Bishop(i, j, 1)
                    case '♝': self.board[j][i] = Bishop(i, j, -1)
                    case '♕': self.board[j][i] = Queen(i, j, 1)
                    case '♛': self.board[j][i] = Queen(i, j, -1)
                    case '♔': self.board[j][i] = King(i, j, 1)
                    case '♚': self.board[j][i] = King(i, j, -1)
                    case '◐': self.board[j][i] = Enpassant(i, j, 1 if j ==2 else -1)
                    case _: self.board[j][i] = Empty(i, j)

    def end_turn(self):
        self.special_move = Special_Move.Normal
        print('  a b c d e f g h')
        for y in range(7, -1, -1):
            print(y + 1, end=' ')
            for x in range(8):
                char = str(self.get_piece(x, y))
                if char == '◐' and y == rank(3, self.turn_color_sign):
                    self.board[y][x] = Empty(x, y)
                    char = ' '
                if char == ' ': char = '□' if (x + y) % 2 == 0 else '■'
                self.display[y][x] = char
                print(char, end='')
            print('', y + 1)
        print('  a b c d e f g h')
        self.turn_color_sign *= -1

class Special_Move(Enum):
    Normal = 0
    Pre_Enpassant = 1
    Enpassant = 2
    Castling = 3
    Promotion = 4

class Piece:
    def __init__(self, x, y, color_sign, char):
        self.x = x
        self.y = y
        self.color_sign = color_sign #white: 1, black: -1
        self.char = char
        self.moved = False

    def move(self, x, y, game):
        game.board[self.y][self.x] = Empty(self.x, self.y)
        game.board[y][x] = self
        self.x = x
        self.y = y

    def __str__(self):
        if (self.color_sign == -1):
            return chr(ord(self.char) + 6)
        else:
            return self.char

    def __mul__(self, other):
        return self.color_sign * other.color_sign

    def is_empty(self): return self.color_sign == 0

    def can_move(self, x, y, game): return False

class Empty(Piece):
    def __init__(self, x, y):
        super().__init__(x, y, 0, ' ')

class Enpassant(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, 0, '◐')

class Pawn(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign, '♙')

    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff_forward = (y - self.y) * self.color_sign
        if (x_diff == 0 and y_diff_forward == 1):
            return isinstance(game.get_piece(x, y), Empty) #move
        if (abs(x_diff) == 1 and y_diff_forward == 1):
            return game.get_piece(x, y).color_sign * self.color_sign == -1 #take-move
        if (x_diff == 0 and y_diff_forward == 2):
            return not self.moved and isinstance(game.get_piece(x, y), Empty)
        return False

class Knight(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign, '♘')
    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff = y - self.y
        return x_diff * x_diff + y_diff * y_diff == 5 and self.color_sign * game.get_piece(x, y).color_sign != 1

class Bishop(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign, '♗')

    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff = y - self.y
        if x_diff == 0 or y_diff == 0: return False
        if game.get_piece(x, y).color_sign * self.color_sign == 1: return False
        slope = y_diff / x_diff
        if (slope == 1):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if game.get_piece(i, i).color_sign != 0: return False
            return True
        if (slope == -1):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if game.get_piece(i, -i).color_sign != 0: return False
            return True
        return False

class Rook(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign, '♖')

    def can_move(self, x, y, game):
        if (x == self.x and y == self.y): return False
        if game.get_piece(x, y).color_sign * self.color_sign == 1: return False
        if (x == self.x):
            for i in range(min(self.y, y) + 1, max(self.y, y)):
                if game.get_piece(x, i).color_sign != 0: return False
            return True
        if (y == self.y):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if game.get_piece(i, y).color_sign != 0: return False
            return True
        return False

class Queen(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign, '♕')

    def can_move(self, x, y, game):
        temp_rook = Rook(self.x, self.y, self.color_sign)
        temp_bishop = Bishop(self.x, self.y, self.color_sign)
        return temp_rook.can_move(x, y, game) or temp_bishop.can_move(x, y, game)

class King(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign, '♔')

    def can_move(self, x, y, game):
        if (abs(self.x - x) > 1 or abs(self.y - y) > 1): return False
        return game.get_piece(x, y).color_sign * self.color_sign != 1



game = Game()
while True:
    try:
        piece = game.input_get()
        if (piece.is_empty()): raise Exception('빈 셀')
        if (piece.color_sign != game.turn_color_sign): raise Exception('상대 기물')
        x, y = game.input_move()
        if (not piece.can_move(x, y, game)): raise Exception('잘못된 행마')
        taken_piece = game.get_piece(x, y)
        if (game.special_move == Special_Move.Enpassant):
            pass#앙파상 처리
        piece.move(x, y, game)
        if (game.special_move == Special_Move.Pre_Enpassant):
            pass#앙파상 이전수 처리
        elif (game.special_move == Special_Move.Castling):
            pass#캐슬링 처리
        if (game.is_self_check()):
            game.undo_move()
            raise Exception('자가 체크')
        if (game.special_move == Special_Move.Promotion):
            pass#프로모션
        game.end_turn();
    except Exception as e:
        print('오류:',e)


  a b c d e f g h
8 ♜♞♝♛♚♝♞♜ 8
7 ♟♟♟♟♟♟♟♟ 7
6 ■□■□■□■□ 6
5 □■□■□■□■ 5
4 ■□■□■□■□ 4
3 □■□■□■□■ 3
2 ♙♙♙♙♙♙♙♙ 2
1 ♖♘♗♕♔♗♘♖ 1
  a b c d e f g h
백 기물 선택: e2
이동할 칸 선택: e4
  a b c d e f g h
8 ♜♞♝♛♚♝♞♜ 8
7 ♟♟♟♟♟♟♟♟ 7
6 ■□■□■□■□ 6
5 □■□■□■□■ 5
4 ■□■□♙□■□ 4
3 □■□■□■□■ 3
2 ♙♙♙♙■♙♙♙ 2
1 ♖♘♗♕♔♗♘♖ 1
  a b c d e f g h
흑 기물 선택: e7
이동할 칸 선택: e5
  a b c d e f g h
8 ♜♞♝♛♚♝♞♜ 8
7 ♟♟♟♟□♟♟♟ 7
6 ■□■□■□■□ 6
5 □■□■♟■□■ 5
4 ■□■□♙□■□ 4
3 □■□■□■□■ 3
2 ♙♙♙♙■♙♙♙ 2
1 ♖♘♗♕♔♗♘♖ 1
  a b c d e f g h


KeyboardInterrupt: Interrupted by user

만들 것

can_move()에서 game.special_move 변경

특수 행마

자살수 금지

1. 턴 시작

2. 기물 좌표 입력 -(오류발생)-> 1로

3. 자신의 기물인가? -거짓-> 1로

4. 이동할 좌표 입력 -(오류발생)-> 1로

5. 이동 가능한가? -거짓-> 1로

6. 앙파상인가? -거짓-> 8로

7. 이동

8. 앙파상 조건을 만드는가? -> 앙파상용 기물 생성

10. 캐슬링인가? -> 룩 이동

11. 자가 체크인가? -참-> 이동 취소 -> 1로

12. 프로모션인가? -참-> 프로모션

13. 턴 종료 -> 1로

폰이 2칸 전진하면 뒤에 투명 기물을 추가해서 앙파상 하게 하자

isinstance 대신 color_sign을 빈 셀 여부 확인용으로 사용 가능

In [ ]:
class Game:
    def __init__(self):
        self.turn_color_sign = 1
        self.board = []
        self._place_pieces(1)
        self._place_pawns(1)
        for i in range(4):
            self.board.append([])
            for j in range(8):
                self.board[i+2].append(None)
        self._place_pawns(-1)
        self._place_pieces(-1)

    def _place_pieces(self, color_sign):
        rank = (7 - 7 * color_sign) // 2
        self.board.append([])
        self.board[rank].append(Rook(0, rank, color_sign))
        self.board[rank].append(Knight(1, rank, color_sign))
        self.board[rank].append(Bishop(2, rank, color_sign))
        self.board[rank].append(Queen(3, rank, color_sign))
        self.board[rank].append(King(4, rank, color_sign))
        self.board[rank].append(Bishop(5, rank, color_sign))
        self.board[rank].append(Knight(6, rank, color_sign))
        self.board[rank].append(Rook(7, rank, color_sign))

    def _place_pawns(self, color_sign):
        rank = (7 - 5 * color_sign) // 2
        self.board.append([])
        for i in range(8):
            self.board[rank].append(Pawn(i, rank, color_sign))

    def print_board(self):
        print('  a b c d e f g h')
        for i in range(7, -1, -1):
            print(i + 1, end=' ')
            for j in range(8):
                print(self._square_char(i, j), end='')
            print('', i + 1)
        print('  a b c d e f g h')

    def _square_char(self, x, y):
        if self.board[x][y] is None:
            if (x + y) % 2 == 0:
                return '□'
            else:
                return '■'
        else:
            return str(self.board[x][y])

    def get_piece(self, x, y):
        if (self.board[y][x]) is None:
            return Empty(x, y)
        return self.board[y][x]

    def move_piece(self, x1, y1, x2, y2):
        piece = self.get_piece(x1, y1)
        if piece.color_sign != self.turn_color_sign:
            print('잘못된 이동입니다. (상대 기물 또는 빈 셀)')
            return False
        if piece.can_move(x2, y2, self):
            piece.move(x2, y2, self)
            return True
        print('잘못된 이동입니다.')
        return False

    def input_move(self):
        while True:
            input_string = input("백: " if self.turn_color_sign else "흑: ")
            if len(input_string) != 4:
                print('잘못된 입력입니다.')
                continue
            try:
                x1 = ord(input_string[0]) - ord('a')
                y1 = int(input_string[1]) - 1
                x2 = ord(input_string[2]) - ord('a')
                y2 = int(input_string[3]) - 1
                if self.move_piece(x1, y1, x2, y2):
                    break
            except:
                print('잘못된 입력입니다.')
        self.turn_color_sign *= -1


class Piece:
    def __init__(self, x, y, color_sign):
        self.x = x
        self.y = y
        self.color_sign = color_sign #white: 1, black: -1
        self.char = ' '
        self.moved = False

    def move(self, x, y, game):
        game.board[self.y][self.x] = None
        game.board[y][x] = self
        self.x = x
        self.y = y
        self.moved = True

    def __str__(self):
        if (self.color_sign == 1):
            return self.char
        else:
            return chr(ord(self.char) + 6)

    def can_move(self, x, y, game):
        return True

class Empty(Piece):
    def __init__(self, x, y):
        super().__init__(x, y, 0)

class Pawn(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♙'

    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff_forward = (y - self.y) * self.color_sign
        if (x_diff == 0 and y_diff_forward == 1):
            return isinstance(game.get_piece(x, y), Empty) #move
        if (abs(x_diff) == 1 and y_diff_forward == 1):
            return game.get_piece(x, y).color_sign * self.color_sign == -1 #take-move
        if (x_diff == 0 and y_diff_forward == 2):
            return not self.moved and isinstance(game.get_piece(x, y), Empty)
        return False

class Knight(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♘'
    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff = y - self.y
        return x_diff * x_diff + y_diff * y_diff == 5 and self.color_sign * game.get_piece(x, y).color_sign != 1

class Bishop(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♗'

    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff = y - self.y
        if x_diff == 0 or y_diff == 0: return False
        if game.board.get_piece(x, y).color_sign * self.color_sign == 1: return False
        slope = y_diff / x_diff
        if (slope == 1):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if game.get_piece(i, i).color_sign != 0: return False
            return True
        if (slope == -1):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if game.get_piece(i, -i).color_sign != 0: return False
            return True
        return False

class Rook(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♖'

    def can_move(self, x, y, game):
        if (x == self.x and y == self.y): return False
        if game.board.get_piece(x, y).color_sign * self.color_sign == 1: return False
        if (x == self.x):
            for i in range(min(self.y, y) + 1, max(self.y, y)):
                if game.get_piece(x, i).color_sign != 0: return False
            return True
        if (y == self.y):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if game.get_piece(i, y).color_sign != 0: return False
            return True
        return False

class Queen(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♕'

    def can_move(self, x, y, game):
        temp_rook = Rook(self.x, self.y, self.color_sign)
        temp_bishop = Bishop(self.x, self.y, self.color_sign)
        return temp_rook.can_move(x, y, game) or temp_bishop.can_move(x, y, game)

class King(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♔'

    def can_move(self, x, y, game):
        if (abs(self.x - x) > 1 or abs(self.y - y) > 1): return False
        return game.get_piece(x, y).color_sign * self.color_sign != 1


game = Game()
game.print_board()
while True:
    game.input_move()
    game.print_board()


  a b c d e f g h
8 ♜♞♝♛♚♝♞♜ 8
7 ♟♟♟♟♟♟♟♟ 7
6 ■□■□■□■□ 6
5 □■□■□■□■ 5
4 ■□■□■□■□ 4
3 □■□■□■□■ 3
2 ♙♙♙♙♙♙♙♙ 2
1 ♖♘♗♕♔♗♘♖ 1
  a b c d e f g h
백: a
잘못된 입력입니다.
백: a
잘못된 입력입니다.
백: e1e1
잘못된 이동입니다.
백: e1e8
잘못된 이동입니다.
백: e9e0
잘못된 입력입니다.


KeyboardInterrupt: Interrupted by user